In [1]:
import re
import emoji
import joblib
import numpy as np

from pythainlp.tokenize import word_tokenize
from scipy.sparse import hstack

In [2]:
def clean_text(text):
    text = str(text)

    text = re.sub(r'&#\d+;', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' <URL> ', text)
    text = re.sub(r'@\S+', ' <USER> ', text)
    text = emoji.replace_emoji(text, replace=' <EMOJI> ')
    text = re.sub(r'[^ก-๙a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def thai_tokenizer(text):
    return word_tokenize(text, engine="newmm")

In [3]:
model = joblib.load("lgb_model.pkl")
tfidf_word = joblib.load("tfidf_word.pkl")
tfidf_char = joblib.load("tfidf_char.pkl")
le = joblib.load("label_encoder.pkl")

In [4]:
def predict_sentiment(text):
    # clean input text
    text = clean_text(text)

    # TF-IDF features
    Xw = tfidf_word.transform([text])
    Xc = tfidf_char.transform([text])

    # combine features
    X = hstack([Xw, Xc])

    # predict
    pred = model.predict(X)

    # convert label back to text
    return le.inverse_transform(pred)[0]

In [5]:
predict_sentiment("รถไฟฟ้าช้ามาก เดือดร้อนสุดๆ")

C:\Users\watta\anaconda3\envs\thai-sentiment\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


'negative'

In [6]:
predict_sentiment("โครงการนี้ช่วยประชาชนได้ดีมาก")

C:\Users\watta\anaconda3\envs\thai-sentiment\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


'negative'

In [7]:
predict_sentiment("วันนี้อากาศร้อน")

C:\Users\watta\anaconda3\envs\thai-sentiment\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


'neutral'

In [8]:
predict_sentiment("นโยบายนี้สร้างปัญหาให้ประชาชน")

C:\Users\watta\anaconda3\envs\thai-sentiment\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


'negative'